# API Demonstration

This notebook demonstrates how the trained CNN model can be exposed through a FastAPI endpoint.

The API accepts an audio file and returns the predicted urban noise category and confidence score.

In [1]:
!pip install fastapi uvicorn python-multipart -q

## Load Required Libraries

In [2]:
import librosa
import numpy as np
import joblib

from tensorflow.keras.models import load_model

## Load Model Assets

In [3]:
MODEL_PATH = "/content/drive/MyDrive/UrbanNoiseProject/models/urban_noise_cnn.keras"

ENCODER_PATH = "/content/drive/MyDrive/UrbanNoiseProject/models/label_encoder.pkl"

model = load_model(MODEL_PATH)

encoder = joblib.load(ENCODER_PATH)

print("Assets Loaded")

Assets Loaded


## Prediction Function

This function processes an audio file and returns a prediction.

In [4]:
def predict_audio(audio_path):

    audio, sr = librosa.load(
        audio_path,
        sr=22050
    )

    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=sr,
        n_mels=128
    )

    mel = librosa.power_to_db(
        mel,
        ref=np.max
    )

    mel = mel[:, :128]

    if mel.shape[1] < 128:
        mel = np.pad(
            mel,
            ((0,0),(0,128-mel.shape[1]))
        )

    X = mel.reshape(
        1,
        128,
        128,
        1
    )

    pred = model.predict(
        X,
        verbose=0
    )

    idx = np.argmax(pred)

    label = encoder.inverse_transform(
        [idx]
    )[0]

    confidence = float(
        np.max(pred)
    )

    return label, confidence

## Simulate API Request

In [5]:
sample_file = "/content/drive/MyDrive/UrbanNoiseProject/datasets/UrbanSound8K/fold1/101415-3-0-2.wav"

label, confidence = predict_audio(
    sample_file
)

print("Prediction:", label)
print("Confidence:", round(confidence*100,2), "%")

Prediction: Ambience
Confidence: 99.97 %


## Example FastAPI Endpoint

The following endpoint would be used in deployment.

In [6]:
api_code = """
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def home():
    return {"message":"Urban Noise API"}

@app.post("/predict")
def predict():
    return {
        "prediction":"Construction",
        "confidence":99.7
    }
}
"""

print(api_code)


from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def home():
    return {"message":"Urban Noise API"}

@app.post("/predict")
def predict():
    return {
        "prediction":"Construction",
        "confidence":99.7
    }
}



## Expected API Response

In [7]:
response = {
    "prediction":"Construction",
    "confidence":99.7
}

response

{'prediction': 'Construction', 'confidence': 99.7}

# Conclusion

The trained CNN model can be integrated into a FastAPI service to classify urban noise recordings.

The API receives an audio file, processes it through the inference pipeline, and returns the predicted category with a confidence score.

This completes the machine learning MVP and prepares the system for deployment.